# Sharing Assets Between Workspaces using regestries


The tutorial is based on the following materials:
- [Machine Learning registries for MLOps](https://learn.microsoft.com/en-us/azure/machine-learning/concept-machine-learning-registries-mlops?view=azureml-api-2)
- [Manage Azure Machine Learning registries](https://learn.microsoft.com/en-us/azure/machine-learning/how-to-manage-registries?view=azureml-api-2&tabs=cli)
- [Share models, components, and environments across workspaces with registries](https://learn.microsoft.com/en-us/azure/machine-learning/how-to-share-models-pipelines-across-workspaces-with-registries?view=azureml-api-2&tabs=python)
- [Share data across workspaces with registries](https://learn.microsoft.com/en-us/azure/machine-learning/how-to-share-data-across-workspaces-with-registries?view=azureml-api-2&tabs=cli)
<!-- - []() -->


# Notebook Setup

Set project paths and load workspace MLClient.

In [2]:
import os
# Here we set the working directory to the project root to ensure imports work correctly
from pathlib import Path
target = "dp100-learn"
p = Path.cwd()
print(f"Starting working directory: {p}")
while p.name != target and p.parent != p:
    p = p.parent
# Set the path to your project root manually if the above code does not work
# p = "/mnt/batch/tasks/shared/LS_root/mounts/clusters/ci-dm-dp100-cpu123814/code/Users/dominik.mika/dp100-learn"
os.chdir(p)
print("Changed working directory to:", p)
from utils.azureml_utils import *

# Get Azure ML Client based on your environment. Learn more in the tutorials/azureml-first-notebook.ipynb.
ml_client = get_azureml_client()

Starting working directory: c:\Users\dmika\DEV\Projects-local\dp100-learn\tutorials
Changed working directory to: c:\Users\dmika\DEV\Projects-local\dp100-learn
Added to sys.path: C:\Users\dmika\DEV\Projects-local\dp100-learn
Added to sys.path: C:\Users\dmika\DEV\Projects-local\dp100-learn


# Create Registry

In [3]:
registry_name = "dmdp100-registry"
registry_location = "westeurope"

In [ ]:
registry_file_path = "assets/tutorials-materials/configs/registry.yml"
parent_image = "mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu20.04"

env_content = f"""
name: {registry_name}
tags:
  description: Basic registry with one primary region and to additional regions
  foo: bar
location: {registry_location}
replication_locations:
  - location: {registry_location}
  - location: swedencentral
"""
with open(registry_file_path, "w") as f:
    f.write(env_content)

In [ ]:
!az ml registry create --resource-group azure-ml-dev-rg --name dmdp100-registry --file assets/tutorials-materials/configs/registry.yml

............{
  "containerRegistry": null,
  "description": null,
  "discoveryUrl": "https://westeurope.api.azureml.ms/registrymanagement/v1.0/registries/dmdp100-registry/discovery",
  "identity": {
    "principalId": "2787d331-6ed6-474c-9ef8-0a1df31c7871",
    "tenantId": "50c76291-0c80-4444-a2fb-4f8ab168c311",
    "type": "SystemAssigned",
    "userAssignedIdentities": null
  },
  "intellectualProperty": null,
  "location": "westeurope",
  "managedResourceGroup": {
    "resourceId": "/subscriptions/a1267753-4c98-48c1-a8e9-9c7169202ffd/resourceGroups/azureml-rg-dmdp100-registry_176eb72d-b721-4444-b2fa-ac9924ed4ace"
  },
  "mlflowRegistryUri": "azureml://westeurope.api.azureml.ms/mlflow/v1.0/subscriptions/a1267753-4c98-48c1-a8e9-9c7169202ffd/resourceGroups/azure-ml-dev-rg/providers/Microsoft.MachineLearningServices/registries/dmdp100-registry",
  "name": "dmdp100-registry",
  "properties": {},
  "publicNetworkAccess": "Enabled",
  "replicationLocations": [
    {
      "acrConfig": [
  

Class RegistryRegionDetailsSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.


# Connect to a Registry

In [4]:
from azure.ai.ml import MLClient

ml_client_registry = MLClient(
    credential=ml_client._credential,
    registry_name=registry_name,
    registry_location=registry_location
)

Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Overriding of current MeterProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented


# Creating Assets

## Creating Assets in Registry

### Create an Environment

In [4]:
env_file_path = "assets/tutorials-materials/configs/registry_env.yml"
main_env_name = "registry_env"
parent_image = "mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu20.04"

env_content = f"""
channels:
  - conda-forge
dependencies:
  - python=3.10.11
  - pip=22.3.1
  - pip:
      - scipy
      - numpy
      - pandas
      - scikit-learn
name: {main_env_name}
"""
with open(env_file_path, "w") as f:
    f.write(env_content)

In [12]:
from azure.ai.ml.entities import Environment

env = Environment(
    name=main_env_name,
    image=parent_image,
    conda_file=env_file_path,
    description="Test environment to share between workspaces usign registries."
)
ml_client_registry.environments.create_or_update(env)

Subtype value SAS has no mapping, use base class DataReferenceCredentialDto.


Environment({'arm_type': 'environment_version', 'latest_version': None, 'image': 'mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu20.04', 'intellectual_property': None, 'is_anonymous': False, 'auto_increment_version': False, 'auto_delete_setting': None, 'name': 'registry_env', 'description': 'Test environment to share between workspaces usign registries.', 'tags': {}, 'properties': {'azureml.labels': 'default,latest,invisibleLatest'}, 'print_as_yaml': False, 'id': 'azureml://registries/dmdp100-registry/environments/registry_env/versions/1', 'Resource__source_path': '', 'base_path': 'c:\\Users\\dmika\\DEV\\Projects-local\\dp100-learn', 'creation_context': <azure.ai.ml.entities._system_data.SystemData object at 0x0000028E543FFA30>, 'serialize': <msrest.serialization.Serializer object at 0x0000028E543DA110>, 'version': '1', 'conda_file': {'channels': ['conda-forge'], 'dependencies': ['python=3.10.11', 'pip=22.3.1', {'pip': ['scipy', 'numpy', 'pandas', 'scikit-learn']}], 'name': 'registry_env

### Register a model

In [ ]:
from azure.ai.ml.entities import Model
from azure.ai.ml.constants import AssetTypes

model_path = "assets/tutorials-materials/models/diabetes-model"
model_name = "diabetes_registry_model"

model = Model(
    name=model_name,
    type=AssetTypes.MLFLOW_MODEL,
    path=model_path,
)
ml_client_registry.models.create_or_update(model)

Subtype value SAS has no mapping, use base class DataReferenceCredentialDto.
Uploading diabetes-model (0.0 MBs): 100%|##########| 2106/2106 [00:00<00:00, 12437.75it/s]




Model({'job_name': None, 'intellectual_property': None, 'system_metadata': None, 'is_anonymous': False, 'auto_increment_version': False, 'auto_delete_setting': None, 'name': 'diabetes_registry_model', 'description': None, 'tags': {}, 'properties': {}, 'print_as_yaml': False, 'id': 'azureml://registries/dmdp100-registry/models/diabetes_registry_model/versions/1', 'Resource__source_path': '', 'base_path': 'c:\\Users\\dmika\\DEV\\Projects-local\\dp100-learn', 'creation_context': <azure.ai.ml.entities._system_data.SystemData object at 0x000001EEE165E650>, 'serialize': <msrest.serialization.Serializer object at 0x000001EEE165E7A0>, 'version': '1', 'latest_version': None, 'path': 'https://a884cf9b451.blob.core.windows.net/dmdp100-re-739ce2ca-5d34-508d-882a-cc6eac76c61f/diabetes-model', 'datastore': None, 'utc_time_created': None, 'flavors': {'python_function': {'env': 'conda.yaml', 'loader_module': 'mlflow.sklearn', 'model_path': 'model.pkl', 'python_version': '3.8.17'}, 'sklearn': {'pickled

### Register Data

In [11]:
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes

my_path = "data/azure-ml-labs-data/diabetes/diabetes.csv"
my_data = Data(
    path=my_path,
    type=AssetTypes.URI_FILE,
    description="Diabetes Data",
    name="diabetes-file",
    version='1'
)
ml_client_registry.data.create_or_update(my_data)

Subtype value SAS has no mapping, use base class DataReferenceCredentialDto.
Uploading diabetes.csv (< 1 MB): 100%|##########| 518k/518k [00:00<00:00, 2.27MB/s]




Data({'path': 'https://a884cf9b451.blob.core.windows.net/dmdp100-re-e80e07ea-fba7-5d9f-b222-424dc9908df5/diabetes.csv', 'skip_validation': False, 'mltable_schema_url': None, 'referenced_uris': None, 'type': 'uri_file', 'is_anonymous': False, 'auto_increment_version': False, 'auto_delete_setting': None, 'name': 'diabetes-file', 'description': 'Diabetes Data', 'tags': {}, 'properties': {}, 'print_as_yaml': False, 'id': 'azureml://registries/dmdp100-registry/data/diabetes-file/versions/1', 'Resource__source_path': '', 'base_path': 'c:\\Users\\dmika\\DEV\\Projects-local\\dp100-learn', 'creation_context': <azure.ai.ml.entities._system_data.SystemData object at 0x0000025C69CB79A0>, 'serialize': <msrest.serialization.Serializer object at 0x0000025C69CB73D0>, 'version': '1', 'latest_version': None, 'datastore': None})

# Share Assets between workspaces

## Connect to other workspaces

In [5]:
from azure.ai.ml import MLClient
from utils.consts import AZUREML_SUBSCRIPTION_ID as subscription_id

ml_client_test_ws = MLClient(
    credential=ml_client._credential,
    subscription_id=subscription_id,
    workspace_name="azure-ml-test-ws",
    resource_group_name="azure-ml-test-rg",
)
ml_client_valid_ws = MLClient(
    credential=ml_client._credential,
    subscription_id=subscription_id,
    workspace_name="azure-ml-valid-ws",
    resource_group_name="azure-ml-test-rg",
)

Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Overriding of current MeterProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Overriding of current MeterProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented


## Deploy model from registry to online endpoint in workspace

In [24]:
from azure.ai.ml.entities import ManagedOnlineEndpoint, ManagedOnlineDeployment
import datetime

# online_endpoint_name = "endpoint-" + datetime.datetime.now().strftime("%m%d%H%M%f")
online_endpoint_name = "endpoint-11261449062675"
endpoint = ManagedOnlineEndpoint(
    name=online_endpoint_name,
    description="this is a sample online endpoint for mlflow model",
    auth_mode="key"
)
# ml_client_test_ws.begin_create_or_update(endpoint)

In [ ]:
from azure.ai.ml.entities import  ManagedOnlineDeployment

mlflow_model_from_registry = ml_client_registry.models.get(name="diabetes_registry_model", version=1)
demo_deployment = ManagedOnlineDeployment(
    name="demo",
    endpoint_name=online_endpoint_name,
    model=mlflow_model_from_registry,
    instance_type="Standard_F4s_v2",
    instance_count=1
)
ml_client_test_ws.online_deployments.begin_create_or_update(demo_deployment)

In [ ]:
endpoint.traffic = {"demo": 100}
ml_client_test_ws.begin_create_or_update(endpoint).begin_create_or_update(endpoint)

## Share a model from workspace to registry

In [14]:
from azure.ai.ml.entities import Model
from azure.ai.ml.constants import AssetTypes

model_path = "assets/tutorials-materials/models/diabetes-model"
model_name = "diabetes_registry_model"

model = Model(
    name=model_name,
    type=AssetTypes.MLFLOW_MODEL,
    path=model_path,
)
ml_client.models.create_or_update(model)

Uploading diabetes-model (0.0 MBs): 100%|##########| 2106/2106 [00:00<00:00, 11694.67it/s]




Model({'job_name': None, 'intellectual_property': None, 'system_metadata': None, 'is_anonymous': False, 'auto_increment_version': False, 'auto_delete_setting': None, 'name': 'diabetes_registry_model', 'description': None, 'tags': {}, 'properties': {}, 'print_as_yaml': False, 'id': '/subscriptions/a1267753-4c98-48c1-a8e9-9c7169202ffd/resourceGroups/azure-ml-dev-rg/providers/Microsoft.MachineLearningServices/workspaces/azure-ml-dev-ws/models/diabetes_registry_model/versions/1', 'Resource__source_path': '', 'base_path': 'c:\\Users\\dmika\\DEV\\Projects-local\\dp100-learn', 'creation_context': <azure.ai.ml.entities._system_data.SystemData object at 0x0000025C1B4FD9C0>, 'serialize': <msrest.serialization.Serializer object at 0x0000025C1B8806D0>, 'version': '1', 'latest_version': None, 'path': 'azureml://subscriptions/a1267753-4c98-48c1-a8e9-9c7169202ffd/resourceGroups/azure-ml-dev-rg/workspaces/azure-ml-dev-ws/datastores/workspaceblobstore/paths/LocalUpload/3f1e2752bc48f144e5c46afa68f3a5faa

In [18]:
# share the model from registry to workspace
ml_client.models.share(
    name="diabetes_registry_model",
    version="1",
    registry_name=registry_name,
    share_with_name="diabetes_registry_model_from_ws",
    share_with_version="1",
)

Model({'job_name': None, 'intellectual_property': None, 'system_metadata': None, 'is_anonymous': False, 'auto_increment_version': False, 'auto_delete_setting': None, 'name': 'diabetes_registry_model_from_ws', 'description': None, 'tags': {}, 'properties': {}, 'print_as_yaml': False, 'id': 'azureml://registries/dmdp100-registry/models/diabetes_registry_model_from_ws/versions/1', 'Resource__source_path': '', 'base_path': 'c:\\Users\\dmika\\DEV\\Projects-local\\dp100-learn', 'creation_context': <azure.ai.ml.entities._system_data.SystemData object at 0x0000025C1BAA3C10>, 'serialize': <msrest.serialization.Serializer object at 0x0000025C1BAA33D0>, 'version': '1', 'latest_version': None, 'path': 'https://a884cf9b451.blob.core.windows.net/dmdp100-re-a500c928-43d3-5612-bbff-007955e8cb56/LocalUpload/3f1e2752bc48f144e5c46afa68f3a5faa65cee349c3809819de09ccd004b0e6f/diabetes-model', 'datastore': None, 'utc_time_created': None, 'flavors': {'python_function': {'env': 'conda.yaml', 'loader_module': '

## Share data from workspace to registry

In [ ]:
# share the model from registry to workspace
ml_client.data.share(
    name="diabetes-file",
    version="1",
    registry_name=registry_name,
    share_with_name="diabetes-file",
    share_with_version="1",
)

In [11]:
from azure.ai.ml.constants import AssetTypes
from azure.ai.ml import Input
from azure.ai.ml import command


registry_data_asset = ml_client_registry.data.get(name="diabetes-file", version=1)
inputs = { "data_input": Input(type=AssetTypes.URI_FILE, path=registry_data_asset.id)}
# configure job
job = command(
    code="assets/tutorials-materials/src",
    inputs=inputs,
    command="ls -R ${{inputs.data_input}}",
    environment="diabetes_registry_model@latest",
    compute="aml-cluster",
    display_name="data_sharing_reg-job-1",
    experiment_name="data_sharing_reg"
)
# submit job
returned_job = ml_client_test_ws.create_or_update(job)
aml_url = returned_job.studio_url
print("Monitor your job at", aml_url)

Monitor your job at https://ml.azure.com/runs/funny_airport_vrfgp6sdbk?wsid=/subscriptions/a1267753-4c98-48c1-a8e9-9c7169202ffd/resourcegroups/azure-ml-test-rg/workspaces/azure-ml-test-ws&tid=50c76291-0c80-4444-a2fb-4f8ab168c311
